# Kaggle S6E3 — Customer Churn v2
Added: ORIG dataset stats + DIGIT features + ROUND features (no slow TE interactions)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')
SEED = 42; N_SPLITS = 5
np.random.seed(SEED)
print('ok')

In [ ]:
import os
if os.path.exists('/kaggle/input'):
    TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e3/train.csv'
    TEST_PATH  = '/kaggle/input/competitions/playground-series-s6e3/test.csv'
    SUB_PATH   = '/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv'
    ORIG_PATH  = '/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv'
else:
    TRAIN_PATH = 'data/train.csv'
    TEST_PATH  = 'data/test.csv'
    SUB_PATH   = 'data/sample_submission.csv'
    ORIG_PATH  = 'data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
orig  = pd.read_csv(ORIG_PATH)
sub   = pd.read_csv(SUB_PATH)
print('train', train.shape, '| test', test.shape, '| orig', orig.shape)

In [ ]:
TARGET='Churn'
CATS=['gender','SeniorCitizen','Partner','Dependents','PhoneService',
    'MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection',
    'TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod']

train_ids=train['id'].copy(); test_ids=test['id'].copy()
train=train.drop(columns=['id']); test=test.drop(columns=['id'])
if 'customerID' in orig.columns: orig=orig.drop(columns=['customerID'])

def map_target(s): return s.map({'Yes':1,'No':0}).astype('int8') if s.dtype==object else s.astype('int8')
train[TARGET]=map_target(train[TARGET]); orig[TARGET]=map_target(orig[TARGET])

for df in [train,test,orig]:
    df['tenure']=pd.to_numeric(df['tenure'],errors='coerce').astype('float32')
    df['MonthlyCharges']=pd.to_numeric(df['MonthlyCharges'],errors='coerce').astype('float32')
    df['TotalCharges']=pd.to_numeric(df['TotalCharges'].astype(str).str.strip().replace('',np.nan),errors='coerce').astype('float32')
    m=df['TotalCharges'].isna()
    df.loc[m,'TotalCharges']=df.loc[m,'tenure']*df.loc[m,'MonthlyCharges']
    for c in CATS:
        if c in df.columns: df[c]=df[c].astype(str).fillna('Missing')

BASE=[c for c in train.columns if c!=TARGET]
print('BASE cols:',len(BASE),'| churn rate:',round(train[TARGET].mean(),4))

In [ ]:
from itertools import combinations

# ── DIGIT features ────────────────────────────────────────────
def add_digit_cols(df, col, multiplier):
    if multiplier == 'direct':
        vals = pd.to_numeric(df[col], errors='coerce').fillna(-1).astype(int)
    else:
        vals = (pd.to_numeric(df[col], errors='coerce') * multiplier).round(0).fillna(-1).astype(int)
    max_len = {'tenure':3,'MonthlyCharges':5,'TotalCharges':6}.get(col, vals.astype(str).str.len().max())
    s = vals.astype(str).str.zfill(max_len)
    created = []
    for i in range(max_len):
        n = f'{col}_DIGIT_{i+1}'; df[n] = s.str[i].astype(int); created.append(n)
    return created

DIGIT = []
for df in [train, test]:
    d = add_digit_cols(df, 'tenure', 'direct')
    add_digit_cols(df, 'MonthlyCharges', 100)
    add_digit_cols(df, 'TotalCharges', 1)
    if df is train:
        DIGIT = d + [f'MonthlyCharges_DIGIT_{i+1}' for i in range(5)] + [f'TotalCharges_DIGIT_{i+1}' for i in range(6)]
print(len(DIGIT), 'DIGIT features')

# ── ROUND features ────────────────────────────────────────────
ROUND = []
for col, levels in [('MonthlyCharges',[('1s',0),('10s',-1),('100s',-2),('1000s',-3)]),
                    ('TotalCharges',  [('1s',0),('10s',-1),('100s',-2),('1000s',-3)]),
                    ('tenure',        [('1s',0),('10s',-1)])]:
    for suffix, level in levels:
        n = f'{col}_ROUND_{suffix}'; ROUND.append(n)
        for df in [train, test]: df[n] = df[col].round(level).fillna(-1).astype(int)
print(len(ROUND), 'ROUND features')

# ── ORIG dataset stats ────────────────────────────────────────
ORIG_FEATS = []
orig_mean = float(orig[TARGET].mean())
for col in BASE:
    mm = orig.groupby(col)[TARGET].mean()
    cm = orig.groupby(col).size()
    for df in [train, test]:
        df[f'orig_mean_{col}']  = df[col].map(mm).fillna(orig_mean)
        df[f'orig_count_{col}'] = df[col].map(cm).fillna(0)
    ORIG_FEATS += [f'orig_mean_{col}', f'orig_count_{col}']
print(len(ORIG_FEATS), 'ORIG features')

# ── INTERACTION features (string concat, TE inside CV) ────────
INTER = []
for col1, col2 in combinations(BASE, 2):
    n = f'{col1}_{col2}'; INTER.append(n)
    for df in [train, test]: df[n] = df[col1].astype(str) + '_' + df[col2].astype(str)
print(len(INTER), 'INTER features')

FEATURES = list(dict.fromkeys(BASE + ORIG_FEATS + INTER + ROUND + DIGIT))
print(len(FEATURES), 'total features')

In [ ]:
from sklearn.model_selection import KFold

def te_encode(X_tr, y_tr, X_val, X_te, cols, n_splits=5):
    """OOF target encode cols, drop originals, return TE columns."""
    gm = float(y_tr.mean())
    full_map = {col: X_tr.assign(_y=y_tr.values).groupby(col)['_y'].mean() for col in cols}
    oof = pd.DataFrame(index=X_tr.index)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    for ti, vi in kf.split(X_tr):
        Xt = X_tr.iloc[ti]; yt = y_tr.iloc[ti]; Xv = X_tr.iloc[vi]
        fm = float(yt.mean())
        for col in cols:
            mp = Xt.assign(_y=yt.values).groupby(col)['_y'].mean()
            oof.loc[Xv.index, f'TE_{col}'] = Xv[col].map(mp).fillna(fm).values
    out_tr = X_tr.copy(); out_val = X_val.copy(); out_te = X_te.copy()
    for col in cols:
        out_tr[f'TE_{col}']  = oof[f'TE_{col}']
        out_val[f'TE_{col}'] = X_val[col].map(full_map[col]).fillna(gm)
        out_te[f'TE_{col}']  = X_te[col].map(full_map[col]).fillna(gm)
    out_tr.drop(columns=cols, inplace=True)
    out_val.drop(columns=cols, inplace=True)
    out_te.drop(columns=cols, inplace=True)
    return out_tr, out_val, out_te

def factorize3(tr, val, te):
    codes, _ = pd.factorize(pd.concat([tr, val, te], ignore_index=True).astype(str))
    n1, n2 = len(tr), len(val)
    return (pd.Series(codes[:n1], index=tr.index, dtype='int32'),
            pd.Series(codes[n1:n1+n2], index=val.index, dtype='int32'),
            pd.Series(codes[n1+n2:], index=te.index, dtype='int32'))

lgb_params = {
    'objective':'binary','metric':'auc','n_estimators':2000,'learning_rate':0.05,
    'num_leaves':127,'subsample':0.8,'colsample_bytree':0.6,'min_child_samples':20,
    'reg_alpha':0.1,'reg_lambda':1.0,'random_state':SEED,'n_jobs':-1,'verbose':-1
}

X = train[FEATURES].copy(); y = train[TARGET].copy(); Xtest = test[FEATURES].copy()
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof = np.zeros(len(X), dtype=np.float32)
tpred = np.zeros(len(Xtest), dtype=np.float32)

for fold, (tri, vali) in enumerate(skf.split(X, y), 1):
    Xtr = X.iloc[tri].copy(); Xval = X.iloc[vali].copy()
    ytr = y.iloc[tri].copy(); yval = y.iloc[vali].copy()
    Xte = Xtest.copy()
    # TE on interaction features (drop string cols, add TE_ numerics)
    Xtr, Xval, Xte = te_encode(Xtr, ytr, Xval, Xte, INTER)
    # factorize remaining categoricals
    for c in CATS:
        if c in Xtr.columns: Xtr[c], Xval[c], Xte[c] = factorize3(Xtr[c], Xval[c], Xte[c])
    for c in Xtr.select_dtypes('object').columns:
        Xtr[c], Xval[c], Xte[c] = factorize3(Xtr[c], Xval[c], Xte[c])
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(Xtr, ytr, eval_set=[(Xval, yval)],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(False)])
    oof[vali] = model.predict_proba(Xval)[:, 1]
    tpred += model.predict_proba(Xte)[:, 1] / N_SPLITS
    print(f'Fold {fold} AUC: {roc_auc_score(yval, oof[vali]):.5f}  best_iter={model.best_iteration_}')

print(f'\nOOF AUC: {roc_auc_score(y, oof):.5f}')

In [ ]:
np.save('oof_lgb_v2.npy', oof); np.save('pred_lgb_v2.npy', tpred)
sub['Churn'] = tpred
sub.to_csv('submission_lgb_v2.csv', index=False)
print('Saved submission_lgb_v2.csv')
sub.head()

In [ ]:
fi = pd.DataFrame({'feature': model.feature_name_, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
plt.figure(figsize=(10,7)); sns.barplot(data=fi.head(25), x='importance', y='feature')
plt.title('Top 25 Features (LightGBM v2)'); plt.tight_layout(); plt.show()